# German Declension Trainer — Level 1: Article + Nomen (All 4 Cases)

**How to use this notebook:**
1. Run **Cell 1** (setup & vocabulary database) once by pressing `Shift + Enter`.
2. Run **Cell 2** to start an interactive session (type `q` to quit anytime, or `stats` to see weak spots).
3. Easily expand `VOCAB` in Cell 1 with nouns from your **Uni Bonn** course or **Instagram** saves!

In [ ]:
import random
from collections import defaultdict

# Format: (gender, singular, plural, n_deklination_singular_ending, n_deklination_genitive_ending,
#          english_singular, english_plural)
# Note: Standard nouns have "" for n_deklination. Masculine weak nouns (n-Deklination) take -n/-en in Akk/Dat/Gen.
VOCAB = [
    ("m", "Tisch", "Tische", "", "", "table", "tables"),
    ("m", "Gedanke", "Gedanken", "n", "ns", "thought", "thoughts"),       # n-Deklination + Genitiv -ns
    ("m", "Student", "Studenten", "en", "en", "student", "students"),     # n-Deklination
    ("m", "Vorschlag", "Vorschläge", "", "", "suggestion", "suggestions"),
    ("m", "Alltag", "Alltage", "", "", "everyday life", "daily routines"),
    ("f", "Entscheidung", "Entscheidungen", "", "", "decision", "decisions"),
    ("f", "Erfahrung", "Erfahrungen", "", "", "experience", "experiences"),
    ("f", "Möglichkeit", "Möglichkeiten", "", "", "possibility", "possibilities"),
    ("f", "Woche", "Wochen", "", "", "week", "weeks"),
    ("f", "Sprache", "Sprachen", "", "", "language", "languages"),
    ("n", "Problem", "Probleme", "", "", "problem", "problems"),
    ("n", "Beispiel", "Beispiele", "", "", "example", "examples"),
    ("n", "Ergebnis", "Ergebnisse", "", "", "result", "results"),
    ("n", "Ziel", "Ziele", "", "", "goal", "goals"),
    ("n", "Jahr", "Jahre", "", "", "year", "years"),
]

# Genitiv Singular forms the rules below can't derive (masc./neuter, strong nouns).
# List every accepted form; the first one is shown as the "main" answer.
# e.g. "Herz": ["Herzens"], "Bus": ["Busses"], "Globus": ["Globus", "Globusses"]
GENITIV_OVERRIDES = {}

ARTICLES = {
    "bestimmt": {
        "Nominativ": {"m": "der", "f": "die", "n": "das", "pl": "die"},
        "Akkusativ": {"m": "den", "f": "die", "n": "das", "pl": "die"},
        "Dativ":     {"m": "dem", "f": "der", "n": "dem", "pl": "den"},
        "Genitiv":   {"m": "des", "f": "der", "n": "des", "pl": "der"},
    },
    "unbestimmt": {
        "Nominativ": {"m": "ein",   "f": "eine",  "n": "ein",   "pl": "keine"},
        "Akkusativ": {"m": "einen", "f": "eine",  "n": "ein",   "pl": "keine"},
        "Dativ":     {"m": "einem", "f": "einer", "n": "einem", "pl": "keinen"},
        "Genitiv":   {"m": "eines", "f": "einer", "n": "eines", "pl": "keiner"},
    },
}

# (German trigger, English translation)
CASE_TRIGGERS = {
    "Nominativ": [("Das Subjekt:", "the subject"), ("Dort steht/sind", "over there is/are"),
                  ("Das ist", "that is")],
    "Akkusativ": [("Ich sehe / brauche", "I see / need"), ("Ohne", "without"), ("Für", "for"),
                  ("Es geht um", "it's about")],
    "Dativ":     [("Mit", "with"), ("Nach", "after"), ("Aus", "out of / from"),
                  ("Bei", "at / at someone's place"), ("Gegenüber", "opposite")],
    "Genitiv":   [("Trotz", "despite"), ("Wegen", "because of"), ("Während", "during"),
                  ("Innerhalb", "within"), ("Aufgrund", "due to")],
}

# Suffixes that only take -s in Genitiv (des Museums, des Mädchens, des Königs, des Teppichs, des Autos)
S_ONLY_ENDINGS = ("um", "chen", "lein", "ling", "ig", "ich", "a", "e", "i", "o", "u", "y")
# Unstressed -el/-en/-er/-em after a consonant also only take -s (des Onkels, des Lehrers, des Problems),
# but NOT after a vowel, where the syllable is stressed (des Ziel(e)s, des Beispiel(e)s, des Bier(e)s)
UNSTRESSED_ENDINGS = ("el", "en", "er", "em")
# s-sounds always take -es (des Hauses, des Fußes, des Reflexes, des Satzes)
ES_ONLY_ENDINGS = ("s", "ß", "x", "z")

def genitive_forms(singular):
    """Accepted Genitiv Singular forms of a strong masculine/neuter noun."""
    if singular in GENITIV_OVERRIDES:
        return list(GENITIV_OVERRIDES[singular])
    low = singular.lower()
    if low.endswith("nis"):                 # Ergebnis -> Ergebnisses (s doubles)
        return [singular + "ses"]
    if low.endswith(ES_ONLY_ENDINGS):
        return [singular + "es"]
    if low.endswith(S_ONLY_ENDINGS):
        return [singular + "s"]
    if low.endswith(UNSTRESSED_ENDINGS) and len(low) > 2 and low[-3] not in "aeiouäöü":
        return [singular + "s"]
    # Everything else: both are correct (des Tisch(e)s, des Jahr(e)s, des Ziel(e)s, des Vorschlag(e)s)
    return [singular + "es", singular + "s"]

def inflect_noun(gender, singular, plural, n_end, n_gen, case, is_plural):
    """Return the list of accepted noun forms (first = main answer)."""
    if is_plural:
        # Dativ Plural always adds -n unless plural already ends in -n or -s
        if case == "Dativ" and not plural.endswith(("n", "s")):
            return [plural + "n"]
        return [plural]

    # Masculine n-Deklination (e.g., der Student -> den Studenten)
    if gender == "m" and n_end:
        if case == "Genitiv":
            return [singular + (n_gen if n_gen else n_end)]
        if case in ("Akkusativ", "Dativ"):
            return [singular + n_end]
        return [singular]

    # Standard Genitiv Singular for masculine & neuter (-s or -es)
    if case == "Genitiv" and gender in ("m", "n"):
        return genitive_forms(singular)

    return [singular]

def generate_item(filter_case=None):
    gender, sg, pl, n_end, n_gen, en_sg, en_pl = random.choice(VOCAB)
    is_plural = random.choice([False, False, True])  # ~33% plural practice
    slot = "pl" if is_plural else gender
    case = filter_case if filter_case else random.choice(list(ARTICLES["bestimmt"].keys()))
    art_type = random.choice(["bestimmt", "unbestimmt"])
    trigger, trigger_en = random.choice(CASE_TRIGGERS[case])

    article = ARTICLES[art_type][case][slot]
    noun_forms = inflect_noun(gender, sg, pl, n_end, n_gen, case, is_plural)
    answers = [f"{article} {form}" for form in noun_forms]

    # Hint tag
    art_label = "bestimmt" if art_type == "bestimmt" else ("negativ (kein-)" if is_plural else "unbestimmt")
    num_label = f"Plural (Sg: {sg})" if is_plural else f"Singular ({'der' if gender=='m' else 'die' if gender=='f' else 'das'} {sg})"
    # English hint, e.g. "(the table, despite)"
    if art_type == "bestimmt":
        en_np = f"the {en_pl if is_plural else en_sg}"
    elif is_plural:
        en_np = f"no {en_pl}"
    else:
        en_np = f"{'an' if en_sg[0] in 'aeiou' else 'a'} {en_sg}"
    prompt = f"[{case} | {art_label} | {num_label}]  →  {trigger} ___   ({en_np}, {trigger_en})"
    return prompt, answers, case, slot

def normalize(text):
    """Collapse extra/leading/trailing whitespace."""
    return " ".join(text.split())

def check_answer(user_ans, answers):
    """Return 'exact', 'case' (right except capitalisation) or None."""
    ans = normalize(user_ans)
    # The article may be capitalised (sentence start); only the noun's capitalisation matters.
    art, _, noun = ans.partition(" ")
    if any(art.casefold() == a_art.casefold() and noun == a_noun
           for a_art, _, a_noun in (a.partition(" ") for a in answers)):
        return "exact"
    if ans.casefold() in (a.casefold() for a in answers):
        return "case"
    return None

SLOT_LABELS = {"m": "maskulin", "f": "feminin", "n": "neutral", "pl": "Plural"}

def new_stats():
    # key -> [correct, total]
    return {"case": defaultdict(lambda: [0, 0]), "slot": defaultdict(lambda: [0, 0])}

def record(stats, case, slot, correct):
    for key, value in (("case", case), ("slot", slot)):
        stats[key][value][1] += 1
        stats[key][value][0] += int(correct)

def print_stats(stats):
    if not stats["case"]:
        print("No answers yet.")
        return
    for key, title in (("case", "By case"), ("slot", "By gender / number")):
        print(f"\n{title} (weakest first):")
        rows = sorted(stats[key].items(), key=lambda kv: kv[1][0] / kv[1][1])
        for value, (right, n) in rows:
            label = SLOT_LABELS.get(value, value)
            print(f"  {label:<10} {right:>3}/{n:<3} ({round(right / n * 100)}%)")

print("Setup complete! Run the next cell to start drilling.")

In [ ]:
# Interactive Drill Loop
# Optional: set FOCUS_CASE = "Genitiv" (or "Nominativ", "Akkusativ", "Dativ") to drill one case only, or None for all 4.
FOCUS_CASE = None

score = 0
total = 0
mistakes = []
stats = new_stats()

print("Type 'q' to finish and see your summary, 'stats' to see your weak spots.\n" + "-" * 55)

prompt, answers, case, slot = generate_item(FOCUS_CASE)
while True:
    print(f"\n{prompt}")
    user_ans = input("Your answer: ").strip()

    if user_ans.lower() in ("q", "quit", "exit"):
        break
    if user_ans.lower() == "stats":
        print_stats(stats)
        continue  # same question again, not counted

    total += 1
    result = check_answer(user_ans, answers)
    record(stats, case, slot, result is not None)
    if result is not None:
        score += 1
        print(f"✅ Richtig! ({score}/{total})")
        if result == "case":
            print(f"   ⚠️ Achte auf die Großschreibung: '{answers[0]}'")
        if len(answers) > 1:
            print(f"   (Beide Formen sind korrekt: {' / '.join(answers)})")
    else:
        correct = " / ".join(answers)
        mistakes.append((prompt, user_ans, correct))
        print(f"❌ Falsch! Dein Input: '{user_ans}'  →  Richtig: '{correct}'")

    prompt, answers, case, slot = generate_item(FOCUS_CASE)

print("\n" + "=" * 55)
if total > 0:
    print(f"Session finished: {score}/{total} correct ({round(score/total*100)}%)")
    print_stats(stats)
    if mistakes:
        print("\nReview these slips:")
        for p, u, e in mistakes:
            print(f"  • {p}\n    You wrote: {u}  |  Correct: {e}")
else:
    print("Session ended.")

## Sentence Mode: Real sentences + case triggers

Loads `sentence_corpus.tsv` (180 sentences with English translations). Fill the gap with **article + noun**.
Each prompt shows the case trigger (verb, preposition, wo?/wohin?, Genitivattribut, …) and the noun in its dictionary form.
Edit the TSV in any spreadsheet app to add your own sentences. Separate alternative answers with `|`.

In [ ]:
# Sentence Mode Drill (needs Cell 1 to be run first)
import csv

FOCUS_CASE = None          # e.g. "Dativ", or None for all cases
FOCUS_TRIGGER_TYPE = None  # e.g. "Wechselpräposition" (substring match), or None
SHOW_ENGLISH_FIRST = False # True = show the translation before you answer

with open("sentence_corpus.tsv", encoding="utf-8") as f:
    CORPUS = list(csv.DictReader(f, delimiter="\t"))

pool = [r for r in CORPUS
        if (FOCUS_CASE is None or r["case"] == FOCUS_CASE)
        and (FOCUS_TRIGGER_TYPE is None or FOCUS_TRIGGER_TYPE.lower() in r["trigger_type"].lower())]
print(f"{len(pool)} sentences loaded. Type 'q' to quit, 'stats' for weak spots.\n" + "-" * 55)

GENDER_OF = {"der": "m", "die": "f", "das": "n"}
score = total = 0
mistakes = []
stats = new_stats()
random.shuffle(pool)

for row in pool:
    answers = [a.strip() for a in row["answer"].split("|")]
    slot = "pl" if row["number"] == "pl" else GENDER_OF[row["noun"].split()[0]]
    num = " (Pl.)" if row["number"] == "pl" else ""
    print(f"\n[{row['trigger']} · {row['trigger_type']}]   ({row['noun']}{num}, {row['article']})")
    print(f"  {row['german']}")
    if SHOW_ENGLISH_FIRST:
        print(f"  {row['english']}")

    while True:
        user_ans = input("Your answer: ").strip()
        if user_ans.lower() == "stats":
            print_stats(stats)
            continue
        break
    if user_ans.lower() in ("q", "quit", "exit"):
        break

    total += 1
    result = check_answer(user_ans, answers)
    record(stats, row["case"], slot, result is not None)
    full = row["german"].replace("___", answers[0])
    if result is not None:
        score += 1
        print(f"✅ Richtig! ({score}/{total})  →  {full}")
        if result == "case":
            print(f"   ⚠️ Achte auf die Großschreibung: '{answers[0]}'")
        if len(answers) > 1:
            print(f"   (Beide Formen sind korrekt: {' / '.join(answers)})")
    else:
        mistakes.append((row["german"], user_ans, " / ".join(answers), row["case"], row["trigger"]))
        print(f"❌ Falsch! ({row['case']})  →  {full}")
    if not SHOW_ENGLISH_FIRST:
        print(f"   {row['english']}")
else:
    print("\n🎉 You went through the whole corpus!")

print("\n" + "=" * 55)
if total:
    print(f"Session finished: {score}/{total} correct ({round(score/total*100)}%)")
    print_stats(stats)
    if mistakes:
        print("\nReview these slips:")
        for s, u, c, case, trig in mistakes:
            print(f"  • {s}  [{case} ← {trig}]\n    You wrote: {u}  |  Correct: {c}")
else:
    print("Session ended.")